In [ ]:
%pip install -r requirements.txt

In [55]:
import os
import pandas as pd
import numpy as np
import seaborn as snsx
import yaml
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from tqdm import tqdm

from src.api_fetcher import TrafficFetcher, SystemFetcher
from src.enriched_data_handler import EnrichedDataLoader

pd.set_option("display.max_columns", 500)

### Mise à jour des infos des capteurs

(nécessaire si le fichier des segments a été modifié)

In [ ]:
sf = SystemFetcher()
sf.sensors_informations(write=True)

### Extraction des dernières informations (V1)

In [2]:
start_date = '2024-11-30 00:00:00Z'
end_date = '2025-05-01 00:00:00Z'

start_day = datetime.strptime(start_date, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')
end_day = datetime.strptime(end_date, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')
traffic = TrafficFetcher(time_start=start_date,time_end=end_date,
                         level="instances")

In [ ]:
traffic_data = traffic.get_all_traffic(waiting_time=5)
traffic_data.to_csv(f"data/extract/{start_day}_{end_day}.csv", index_label=False)

### Extraction des dernières informations (V2)

In [174]:
start_date_period = '2023-08-01 00:00:00Z'
end_date_period = '2023-09-01 00:00:00Z'

start_day = datetime.strptime(start_date_period, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')
end_day = datetime.strptime(end_date, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')

In [175]:
with open('config/sensors.yaml', 'r') as file:
    sensors = yaml.safe_load(file)
sensors = pd.DataFrame(sensors).T

sensors['time_added'] = pd.to_datetime(sensors['time_added'])
sensors['first_data_package'] = pd.to_datetime(sensors['first_data_package'])
sensors['last_data_package'] = pd.to_datetime(sensors['last_data_package'])

sensors_v2 = list(sensors[(sensors['hardware_version'] == 2) &
                          (sensors['last_data_package'] >= start_day) &
                          (sensors['first_data_package'] <= end_day)].instance_id.unique())
sensors_done = []

In [176]:
todo_sensors = list(set(sensors_v2) - set(sensors_done))
todo_sensors.sort()
todo_sensors

[7782, 7783, 7785, 7790, 7803]

In [177]:
for sensor in tqdm(todo_sensors):
    first_data_package = [sensors[sensors['instance_id'] == sensor]['first_data_package'] + pd.Timedelta(days=1)][0].dt.strftime('%Y-%m-%d 00:00:00Z').values[0]
    last_data_package = [sensors[sensors['instance_id'] == sensor]['last_data_package'] + pd.Timedelta(days=1)][0].dt.strftime('%Y-%m-%d 00:00:00Z').values[0]
    
    start_date = max(start_date_period, first_data_package)
    start_day = datetime.strptime(start_date, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')

    end_date = min(end_date_period, last_data_package)
    end_day = datetime.strptime(end_date, '%Y-%m-%d %H:%M:%SZ').strftime('%Y%m%d')

    print(f"sensor: {sensor}, start_date:{start_date}, end_date:{end_date}")
    traffic = TrafficFetcher(time_start=start_date,time_end=end_date,
                         level="instances", advanced=True,
                         telraam_format="per-quarter")
    traffic_data = traffic.get_traffic(sensor)
    if len(traffic_data) > 0:
        traffic_data.to_csv(f"data/v2/{start_day}_{end_day}_{sensor}.csv", index_label=False)
    sensors_done.append(sensor)

100%|██████████| 5/5 [00:00<00:00, 483.24it/s]

sensor: 7782, start_date:2023-09-08 00:00:00Z, end_date:2023-09-01 00:00:00Z
7782
sensor: 7783, start_date:2023-09-08 00:00:00Z, end_date:2023-09-01 00:00:00Z
7783
sensor: 7785, start_date:2023-09-08 00:00:00Z, end_date:2023-09-01 00:00:00Z
7785
sensor: 7790, start_date:2023-09-08 00:00:00Z, end_date:2023-09-01 00:00:00Z
7790
sensor: 7803, start_date:2023-09-11 00:00:00Z, end_date:2023-09-01 00:00:00Z
7803


### Concaténer les fichiers

In [191]:
# V2
data_loader = EnrichedDataLoader(directory="data/v2")
raw_data = data_loader.get_raw_data()
enriched_data = data_loader.get_enriched_data()
enriched_data = enriched_data.drop(columns=['segment_id_x']).rename(columns={"segment_id_y": "segment_id"})

min_date = enriched_data['date'].min().strftime("%Y%m%d")
max_date = enriched_data['date'].max().strftime("%Y%m%d")
enriched_data.to_csv(f'data/results/v2_{min_date}_{max_date}_sensors_extract.csv', index_label=False)

In [190]:
# V1
data_loader = EnrichedDataLoader(directory="data/")
raw_data = data_loader.get_raw_data()
enriched_data = data_loader.get_enriched_data()
enriched_data = enriched_data.drop(columns=['segment_id_x']).rename(columns={"segment_id_y": "segment_id"})

min_date = enriched_data['date'].min().strftime("%Y%m%d")
max_date = enriched_data['date'].max().strftime("%Y%m%d")
enriched_data.to_csv(f'data/results/v1_{min_date}_{max_date}_sensors_extract.csv', index_label=False)

### MISC : coller les infos des fichiers yml

In [ ]:
with open('config/sensors.yaml', 'r') as file:
    sensors = yaml.safe_load(file)
instances = [str(sensors[info]['instance_id']) for info in sensors]
','.join(instances)